In [3]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd

# ============================================================
# CONFIG: update if your filenames differ
# ============================================================
BASE_DIR = Path(
    r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention"
)
OUTPUT_DIR = BASE_DIR / "Output"

# Episode-level output (one row per episode, with start_* and end_* intention columns)
EPISODES_WITH_INTENTIONS = OUTPUT_DIR / "2_All_episodes_with_messages_with_intentions_subcat.csv"

# Commit-level output (one row per boundary commit, with intent_* columns)
COMMITS_WITH_INTENTIONS = OUTPUT_DIR / "1_All_Commits_PR_Msg_Iss_with_intentions_subcat.csv"

# Where to save summary tables
OUT_OBS1_EP = OUTPUT_DIR / "OBS1_summary_unique_boundary_commits_from_episodes.csv"
OUT_OBS1_CM = OUTPUT_DIR / "OBS1_summary_unique_boundary_commits_from_commits.csv"

OUT_OBS2_STYLE_ADD = OUTPUT_DIR / "OBS2_style_breakdown_added_unique_commits.csv"
OUT_OBS2_STYLE_REM = OUTPUT_DIR / "OBS2_style_breakdown_removed_unique_commits.csv"
OUT_OBS2_STYLE_ADD_RELEASE = OUTPUT_DIR / "OBS2_release_workflows_rate_by_style_added.csv"

# If you want multi-label counting (each commit can contribute to multiple labels),
# set this True. If False, only the FIRST label in "A || B || C" is used.
COUNT_MULTI_LABELS = False

LABEL_SEP = " || "  # matches your labeling output
EVENT_ADDED = "added"
EVENT_REMOVED = "removed"


# ============================================================
# Helpers
# ============================================================
def _safe_str(x) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    return "" if s.lower() == "nan" else s


def _split_labels(label_str: str) -> List[str]:
    s = _safe_str(label_str)
    if not s:
        return []
    parts = [p.strip() for p in s.split(LABEL_SEP)]
    return [p for p in parts if p]


def _pick_primary_label(label_str: str) -> str:
    labs = _split_labels(label_str)
    return labs[0] if labs else ""


def _extract_side_from_joined(joined: str, side_index: int) -> str:
    """
    Episodes file often stores boundary fields as:
      - "added" (only start exists or both same)
      - "added || removed" (start, end)
    side_index: 0 for start, 1 for end
    """
    s = _safe_str(joined)
    if not s:
        return ""
    parts = [p.strip() for p in s.split(LABEL_SEP)]
    if len(parts) == 1:
        return parts[0]  # fallback
    return parts[side_index] if side_index < len(parts) else parts[-1]


def _dedup_keep_best_label(df: pd.DataFrame, key_cols: List[str], label_col: str) -> pd.DataFrame:
    """
    Deduplicate by key_cols while preferring rows that have a non-empty label.
    If multiple rows have labels, keep the first (stable).
    """
    df = df.copy()
    df["_has_label"] = df[label_col].map(lambda x: 1 if _safe_str(x) else 0)
    df = df.sort_values(by=key_cols + ["_has_label"], ascending=[True] * len(key_cols) + [False])
    df = df.drop_duplicates(subset=key_cols, keep="first").drop(columns=["_has_label"])
    return df


def _detection_rate(df: pd.DataFrame, label_col: str) -> Tuple[int, int, float]:
    n_total = len(df)
    n_labeled = int((df[label_col].map(_safe_str) != "").sum())
    rate = (n_labeled / n_total) if n_total else 0.0
    return n_total, n_labeled, rate


def _top_intentions_table(
    df: pd.DataFrame,
    label_col: str,
    event_type_col: str,
    event_value: str,
    top_k: int = 10,
) -> pd.DataFrame:
    """
    Returns a table with counts + % among LABELED commits only,
    for a specific subset like event_type == 'added' or 'removed'.
    """
    sub = df[df[event_type_col].map(_safe_str).str.lower() == event_value].copy()

    # only labeled commits contribute to intention distribution
    sub_labeled = sub[sub[label_col].map(_safe_str) != ""].copy()
    denom = len(sub_labeled)

    if denom == 0:
        return pd.DataFrame(columns=["event_type", "intention", "count_labeled", "pct_labeled"])

    if COUNT_MULTI_LABELS:
        rows = []
        for s in sub_labeled[label_col]:
            for lab in _split_labels(s):
                rows.append(lab)
        counts = pd.Series(rows).value_counts()
    else:
        counts = sub_labeled[label_col].map(_pick_primary_label).value_counts()

    out = counts.head(top_k).reset_index()
    out.columns = ["intention", "count_labeled"]
    out["pct_labeled"] = out["count_labeled"] / float(denom)
    out.insert(0, "event_type", event_value)
    return out


# ============================================================
# Build unique boundary-commit dataset from EPISODES file
# ============================================================
def boundary_commits_from_episodes(episodes: pd.DataFrame) -> pd.DataFrame:
    """
    Flatten episodes rows into boundary-commit rows:
      - one row for each start boundary
      - one row for each end boundary (if end exists)

    Output columns:
      repo_name, commit_sha, boundary_side, env_style, event_type, label_str, confidence_level
    """
    rows = []

    for _, r in episodes.iterrows():
        repo = _safe_str(r.get("repo_name", ""))
        if not repo:
            continue

        # --- START ---
        start_sha = _safe_str(r.get("episode_start_commit_sha", ""))
        if start_sha:
            rows.append(
                {
                    "repo_name": repo,
                    "commit_sha": start_sha,
                    "boundary_side": "start",
                    # env_styles/event_types might be joined "X || Y"
                    "env_style": _extract_side_from_joined(r.get("env_styles", ""), 0),
                    "event_type": _extract_side_from_joined(r.get("boundary_event_types", ""), 0).lower(),
                    "label_str": _safe_str(r.get("start_label_str", "")),
                    "confidence_level": _safe_str(r.get("start_confidence_level", "")),
                }
            )

        # --- END ---
        end_sha = _safe_str(r.get("episode_end_commit_sha", ""))
        if end_sha:
            rows.append(
                {
                    "repo_name": repo,
                    "commit_sha": end_sha,
                    "boundary_side": "end",
                    "env_style": _extract_side_from_joined(r.get("env_styles", ""), 1),
                    "event_type": _extract_side_from_joined(r.get("boundary_event_types", ""), 1).lower(),
                    "label_str": _safe_str(r.get("end_label_str", "")),
                    "confidence_level": _safe_str(r.get("end_confidence_level", "")),
                }
            )

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    # Deduplicate commits across episodes (same commit can appear as end then start, etc.)
    out = _dedup_keep_best_label(out, key_cols=["repo_name", "commit_sha"], label_col="label_str")

    # Normalize event_type to {added, removed, other}
    out["event_type"] = out["event_type"].map(lambda x: _safe_str(x).lower())
    return out


# ============================================================
# Build unique boundary-commit dataset from COMMITS file
# ============================================================
def boundary_commits_from_commits(commits: pd.DataFrame) -> pd.DataFrame:
    """
    Use the commit-level file directly, but deduplicate by (repo_name, commit_sha).
    Expected columns (typical):
      repo_name, commit_sha, env_style, event_type, intent_label_str, intent_confidence_level
    """
    df = commits.copy()

    # Normalize column names (support small variations)
    if "commit_sha" not in df.columns and "commit" in df.columns:
        df["commit_sha"] = df["commit"]

    # Keep only what we need
    needed = {
        "repo_name": "repo_name",
        "commit_sha": "commit_sha",
        "env_style": "env_style",
        "event_type": "event_type",
        "intent_label_str": "label_str",
        "intent_confidence_level": "confidence_level",
    }
    for src, dst in needed.items():
        if src in df.columns:
            df[dst] = df[src]
        else:
            df[dst] = ""

    out = df[["repo_name", "commit_sha", "env_style", "event_type", "label_str", "confidence_level"]].copy()

    out["repo_name"] = out["repo_name"].map(_safe_str)
    out["commit_sha"] = out["commit_sha"].map(_safe_str)
    out["env_style"] = out["env_style"].map(_safe_str)
    out["event_type"] = out["event_type"].map(lambda x: _safe_str(x).lower())
    out["label_str"] = out["label_str"].map(_safe_str)
    out["confidence_level"] = out["confidence_level"].map(_safe_str)

    out = out[(out["repo_name"] != "") & (out["commit_sha"] != "")].copy()

    # Deduplicate boundary commits (avoid inflation)
    out = _dedup_keep_best_label(out, key_cols=["repo_name", "commit_sha"], label_col="label_str")
    return out


# ============================================================
# Observation 1 stats generator
# ============================================================
def make_observation_1_summary(unique_boundary: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    """
    Observation 1:
      - overall detection rate on UNIQUE boundary commits
      - top intentions for added vs removed (among LABELED unique commits)
    """
    n_total, n_labeled, rate = _detection_rate(unique_boundary, label_col="label_str")

    # Overall row
    summary_rows = [
        {
            "dataset": dataset_name,
            "scope": "overall_unique_boundary_commits",
            "unique_commits_total": n_total,
            "unique_commits_labeled": n_labeled,
            "detection_rate": rate,
        }
    ]

    # Added / removed detection rates
    for ev in (EVENT_ADDED, EVENT_REMOVED):
        sub = unique_boundary[unique_boundary["event_type"] == ev]
        st, sl, sr = _detection_rate(sub, label_col="label_str")
        summary_rows.append(
            {
                "dataset": dataset_name,
                "scope": f"{ev}_unique_boundary_commits",
                "unique_commits_total": st,
                "unique_commits_labeled": sl,
                "detection_rate": sr,
            }
        )

    summary = pd.DataFrame(summary_rows)

    # Add top intention tables as separate prints/CSVs from main()
    return summary


# ============================================================
# Observation 2 stats generator (style-specific)
# ============================================================
def make_observation_2_style_tables(unique_boundary: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Observation 2 (style-specific):
      - For each env_style, show labeled distribution for ADDED and REMOVED
      - Additionally, show the "Automate or integrate release workflows" rate per env_style for ADDED
    """
    df = unique_boundary.copy()
    df["primary_label"] = df["label_str"].map(_pick_primary_label)

    # Only labeled commits for intention mix
    labeled = df[df["label_str"] != ""].copy()

    # --- style x intention distribution (ADDED) ---
    add = labeled[labeled["event_type"] == EVENT_ADDED].copy()
    add_tab = (
        add.groupby(["env_style", "primary_label"])
        .size()
        .reset_index(name="count_labeled")
        .sort_values(["env_style", "count_labeled"], ascending=[True, False])
    )
    add_tot = add.groupby("env_style").size().reset_index(name="labeled_total")
    add_tab = add_tab.merge(add_tot, on="env_style", how="left")
    add_tab["pct_within_style_labeled"] = add_tab["count_labeled"] / add_tab["labeled_total"]

    # --- style x intention distribution (REMOVED) ---
    rem = labeled[labeled["event_type"] == EVENT_REMOVED].copy()
    rem_tab = (
        rem.groupby(["env_style", "primary_label"])
        .size()
        .reset_index(name="count_labeled")
        .sort_values(["env_style", "count_labeled"], ascending=[True, False])
    )
    rem_tot = rem.groupby("env_style").size().reset_index(name="labeled_total")
    rem_tab = rem_tab.merge(rem_tot, on="env_style", how="left")
    rem_tab["pct_within_style_labeled"] = rem_tab["count_labeled"] / rem_tab["labeled_total"]

    # --- release workflow rate by style for ADDED (among labeled commits for that style) ---
    release_label = "Automate or integrate release workflows"
    add_release = add_tot.copy()
    add_release = add_release.rename(columns={"labeled_total": "added_labeled_total"})

    rel_counts = (
        add[add["primary_label"] == release_label]
        .groupby("env_style")
        .size()
        .reset_index(name="release_workflows_count")
    )
    add_release = add_release.merge(rel_counts, on="env_style", how="left")
    add_release["release_workflows_count"] = add_release["release_workflows_count"].fillna(0).astype(int)
    add_release["release_workflows_rate_labeled"] = (
        add_release["release_workflows_count"] / add_release["added_labeled_total"]
    )

    add_release = add_release.sort_values("release_workflows_rate_labeled", ascending=False)

    return add_tab, rem_tab, add_release


# ============================================================
# Main
# ============================================================
def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # --------------------------
    # Load both datasets
    # --------------------------
    if not EPISODES_WITH_INTENTIONS.exists():
        raise FileNotFoundError(f"Missing: {EPISODES_WITH_INTENTIONS}")
    if not COMMITS_WITH_INTENTIONS.exists():
        raise FileNotFoundError(f"Missing: {COMMITS_WITH_INTENTIONS}")

    ep = pd.read_csv(EPISODES_WITH_INTENTIONS, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")
    cm = pd.read_csv(COMMITS_WITH_INTENTIONS, dtype=str, keep_default_na=False, encoding="utf-8", engine="python")

    # --------------------------
    # Build UNIQUE boundary commits (no inflation)
    # --------------------------
    ep_unique = boundary_commits_from_episodes(ep)
    cm_unique = boundary_commits_from_commits(cm)

    # --------------------------
    # Observation 1 summaries
    # --------------------------
    obs1_ep = make_observation_1_summary(ep_unique, dataset_name="episodes_flattened")
    obs1_cm = make_observation_1_summary(cm_unique, dataset_name="commits_list")

    obs1_ep.to_csv(OUT_OBS1_EP, index=False, encoding="utf-8")
    obs1_cm.to_csv(OUT_OBS1_CM, index=False, encoding="utf-8")

    # Top intentions tables (added/removed) for both datasets
    # Episodes-derived
    ep_top_added = _top_intentions_table(ep_unique, "label_str", "event_type", EVENT_ADDED, top_k=10)
    ep_top_removed = _top_intentions_table(ep_unique, "label_str", "event_type", EVENT_REMOVED, top_k=10)

    # Commits-derived
    cm_top_added = _top_intentions_table(cm_unique, "label_str", "event_type", EVENT_ADDED, top_k=10)
    cm_top_removed = _top_intentions_table(cm_unique, "label_str", "event_type", EVENT_REMOVED, top_k=10)

    # Save top tables
    ep_top_added.to_csv(OUTPUT_DIR / "OBS1_top_intentions_added_from_episodes.csv", index=False, encoding="utf-8")
    ep_top_removed.to_csv(OUTPUT_DIR / "OBS1_top_intentions_removed_from_episodes.csv", index=False, encoding="utf-8")
    cm_top_added.to_csv(OUTPUT_DIR / "OBS1_top_intentions_added_from_commits.csv", index=False, encoding="utf-8")
    cm_top_removed.to_csv(OUTPUT_DIR / "OBS1_top_intentions_removed_from_commits.csv", index=False, encoding="utf-8")

    # --------------------------
    # Observation 2 (style-specific) from commit-level unique dataset
    # (This is the stronger source because it has env_style per boundary commit row.)
    # --------------------------
    add_tab, rem_tab, add_release = make_observation_2_style_tables(cm_unique)
    add_tab.to_csv(OUT_OBS2_STYLE_ADD, index=False, encoding="utf-8")
    rem_tab.to_csv(OUT_OBS2_STYLE_REM, index=False, encoding="utf-8")
    add_release.to_csv(OUT_OBS2_STYLE_ADD_RELEASE, index=False, encoding="utf-8")

    # --------------------------
    # Console output (short)
    # --------------------------
    def _print_obs1(summary: pd.DataFrame, tag: str) -> None:
        overall = summary[summary["scope"] == "overall_unique_boundary_commits"].iloc[0]
        print(f"\n[{tag}] Overall detection rate (unique boundary commits): "
              f"{overall['unique_commits_labeled']}/{overall['unique_commits_total']} = {overall['detection_rate']:.3f}")

        for ev in (EVENT_ADDED, EVENT_REMOVED):
            row = summary[summary["scope"] == f"{ev}_unique_boundary_commits"].iloc[0]
            print(f"  - {ev}: {row['unique_commits_labeled']}/{row['unique_commits_total']} = {row['detection_rate']:.3f}")

    _print_obs1(obs1_ep, "episodes_flattened")
    _print_obs1(obs1_cm, "commits_list")

    print("\n[ok] Wrote Observation 1 summaries:")
    print("  -", OUT_OBS1_EP)
    print("  -", OUT_OBS1_CM)

    print("\n[ok] Wrote Observation 2 style tables (from commits_list):")
    print("  -", OUT_OBS2_STYLE_ADD)
    print("  -", OUT_OBS2_STYLE_REM)
    print("  -", OUT_OBS2_STYLE_ADD_RELEASE)

    print("\n[ok] Also wrote top-intention tables for added/removed (both datasets).")


if __name__ == "__main__":
    main()



[episodes_flattened] Overall detection rate (unique boundary commits): 283/535 = 0.529
  - added: 268/515 = 0.520
  - removed: 15/20 = 0.750

[commits_list] Overall detection rate (unique boundary commits): 283/535 = 0.529
  - added: 268/515 = 0.520
  - removed: 15/20 = 0.750

[ok] Wrote Observation 1 summaries:
  - C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\OBS1_summary_unique_boundary_commits_from_episodes.csv
  - C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\OBS1_summary_unique_boundary_commits_from_commits.csv

[ok] Wrote Observation 2 style tables (from commits_list):
  - C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\OBS2_style_breakdown_added_unique_commits.csv
  - C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\OBS2_style_breakdown_removed_u